<a href="https://colab.research.google.com/github/Diego-LeivaC/market-leiva/blob/main/market-notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The following notebook describes....

## **Data Loading**

In [2]:
import os
os.environ['KAGGLE_USERNAME'] = "xxxxx"
os.environ['KAGGLE_KEY'] = "xxxxx"
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows


Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
  0% 0.00/175k [00:00<?, ?B/s]
100% 175k/175k [00:00<00:00, 305MB/s]


In [3]:
!unzip *.zip

Archive:  imdb-dataset-of-top-1000-movies-and-tv-shows.zip
  inflating: imdb_top_1000.csv       


In [4]:
import pandas as pd

data = pd.read_csv("imdb_top_1000.csv")
data.head(3)

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"


# **Data Analysis and Preprocessing**

In [5]:
data.shape

(1000, 16)

There are 1000 rows (1000 films) and 16 columns.

In [6]:
data.columns

Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='object')

The column 'Series_Title' refers to the film's name.

In [7]:
data['Series_Title'].isnull().sum()

np.int64(0)

All films are completed.

The columns: 'Star1', 'Star2', 'Star3', 'Star4' are the actor names.

In [8]:
data[['Star1','Star2','Star3','Star4']].isnull().sum()

,0
Star1,0
Star2,0
Star3,0
Star4,0


All actors are completed.

In [9]:
data = data.drop_duplicates()
data.shape

(1000, 16)

There are no duplicates.

In [10]:
for col in ["Star1","Star2","Star3","Star4"]:
    data[col] = data[col].str.strip().str.lower()

To uniformize the actor's names, I remove spaces and put it in lowercase.

# **The Market-Basket Model**

*"The market-basket model of data is used to describe a common form of many
many relationship between two kinds of objects. On the one hand, we have
items, and on the other we have baskets, sometimes called “transactions.”
Each basket consists of a set of items (an itemset), and usually we assume that
the number of items in a basket is small– much smaller than the total number
of items. The numbe *testo in corsivo*r of baskets is usually assumed to be very large, bigger than what can fit in main memory."* (book 6.1)

Considering that each film is to be considered a basket, and actors listed under the Star1, Star2, Star3 and Star4 fields as items. I transform each film in a list of actors. testo in grassetto

In [11]:
transactions = data[['Star1','Star2','Star3','Star4']].values.tolist()

In [12]:
import random
print(random.sample(transactions, 2))

[['danny glover', 'whoopi goldberg', 'oprah winfrey', 'margaret avery'], ['jesse eisenberg', 'emma stone', 'woody harrelson', 'abigail breslin']]


The attribute transactions is a market-basket format.

Before passing the A-priori algorithm, considered these concepts associated with the project:

*   Itemset Size ($k$): Represents the number of items contained within a specific candidate or frequent set during a given iteration. For this project, $k$ defines the number of actors being analyzed in a combination, where the algorithm progresses from individual actors ($k=1$) to pairs ($k=2$), triples ($k=3$), until $k=4$.

*   Support Threshold ($s$): Is the minimum frequency that an itemset must reach to be considered significant and "frequent". In this context, $s$ determines the minimum number of movies in which a group of actors must appear together for their relationship to be identified as a pattern.





# **A-priori Algorithm for All Frequent Itemset**

*"In the A-Priori Algorithm, one pass is taken for each set-size k. If no frequent
itemsets of a certain size are found, then monotonicity tells us there can be no
larger frequent itemsets, so we can stop.
The pattern of moving from one size k to the next size k + 1 can be summarized as follows. For each size k, there are two sets of itemsets: Ck is the set of candidate itemsets of size k– the itemsets that we must
count in order to determine whether they are in fact frequent. Lk is the set of truly frequent itemsets of size k."*(book 15)

In practice, the mlxtend library runs the full A-Priori algorithm. This algorithm operates through iterative cycles (multiple passes):



* Pass 1: Finds frequent individual items ($L_1$) who appear in more than $s$ films, where $s$ is the support threshold.


*   Pass 2: Combines items from the previous step to find frequent pairs ($L_2$).

*   Pass 3: Takes the frequent pairs from step 2 and combines them to try to form candidate triplets ($C_3$). If those triplets exceed the minimum support, they become frequent triplets ($L_3$).

*   Pass 4: Generates candidate quadruplets ($C_4$). This is the maximum limit. The algorithm will attempt to see whether there are 4 actors who always appear together in the 4 columns in a film.

End of the Algorithm: The algorithm stops because there is no basket with 5 actors, so the support of any quintuplet would automatically be 0.


In [13]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

transaction_encoder = TransactionEncoder() # Creates an object that can transform data (actor names) to a mathematic format.
te_array = transaction_encoder.fit(transactions).transform(transactions) # Convert the variable "transactions", transforming it into a boolean matrix False/True.
data_encoded = pd.DataFrame(te_array, columns=transaction_encoder.columns_) # Convert the matrix into a table

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
data_encoded.head(3)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,aamir bashir,aamir khan,aaron eckhart,aaron taylor-johnson,abdel ahmed ghili,abhay deol,abigail breslin,abraham attah,adam baldwin,adam driver,...,ziyi zhang,zoe saldana,zooey deschanel,zoë kravitz,álvaro guerrero,çetin tekindor,émile vallée,éric toledano,ömer faruk sorak,özge özberk
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


**Interpretation:** "True" means that the actor appears in the film, "False" means that not. For instance, among the 2709 actors, Aaron Eckhart was the star 3 of the film The Dark Knight, this film was in the row 2 on the original dataset.

*"After the first pass, we examine the counts of the items to determine which of them are frequent as singletons. It might appear surprising that many singletons are not frequent. But remember that we set the threshold s sufficiently high that we do not get too many frequent sets; a typical s would be 1% of the baskets."* (book 14)

In [ ]:
frequent_combinations = apriori(data_encoded, min_support=0.01, use_colnames=True)
frequent_combinations.shape

(9, 2)

In [ ]:
frequent_combinations.sort_values("support", ascending=False).head(9) # Orders by frequency, most frequent actors of the dataset, to see also the maximun support

,support,itemsets
7,0.017,(robert de niro)
8,0.014,(tom hanks)
0,0.013,(al pacino)
1,0.012,(brad pitt)
3,0.012,(clint eastwood)
6,0.011,(matt damon)
2,0.011,(christian bale)
5,0.011,(leonardo dicaprio)
4,0.010,(james stewart)


**Interpretation:** The maximum support threshold is 0.017.
Robert De Niro appears in 1.7% of films, meaning that De Niro appears in 17 of the 1000 films in the dataset. The same logic applies to the rest of actors.

Testing different support threshold

In [ ]:
supports = [0.017, 0.005, 0.002, 0.0012]

results = []
for s in supports:
    res = apriori(data_encoded, min_support=s, use_colnames=True)
    results.append((s, len(res)))

pd.DataFrame(results, columns=["support","num_itemsets"])

,support,num_itemsets
0,0.0170,1
1,0.0050,83
2,0.0020,791
3,0.0012,791


**Interpretation:** As expected only one itemset (Robert De Niro) has the support 0.017, as the support threshold decreases the number of itemsets increases, where 0.0020 and 0.0012 have the same number since 1.2 films is equivalent to 2 films because the algorihm works with whole numbers.

# **Main memory limitation**

When the support threshold was lowered to 0.001, the algorithm failed due to memory limitations with this message: "The session crashed because all available RAM was used".
With support=0.001, the algorithm returns all itemsets that appear in at least 1 film and that involves practically every possible combination of actors.

To find the frequent pairs, the algorithm follows these steps according to the theory:

*   Step 1: Identify frequent items ($L_1$): First, it finds which actors appear in more than $s$ films. With $s=0.001$, all 2,709 actors are considered frequent.

*   Step 2: Generate Candidates ($C_2$): This is where the memory disaster could occurs. The algorithm generates all possible combinations of the frequent actors to create a data structure where it will accumulate the counts.


*   Combinatorial Analysis: Since the actors are represented as columns (2,709 columns), A-Priori generates the combination of every column with one another. This means $\binom{2709}{2} \approx 3.6$ million candidate pairs (exactly 3,667,986 pairs). For k=3 and k=4 the candidate pairs are also higher.



# **Analysis of Multi-Actor Itemsets ($k \geq 2$) with $s=0.002$**

The next code block filters the results to show combinations of two or more actors, reducing the support threshold to 0.002 (which means it's enough that they've appeared together in 2 films)

In [ ]:
warnings.filterwarnings('ignore', category=DeprecationWarning)
frequent_combinations = apriori(data_encoded, min_support=0.002, use_colnames=True)

# Creation of a column with the length of each combination
frequent_combinations['length'] = frequent_combinations['itemsets'].apply(lambda x: len(x))

# Filter to show only combinations of 2 or more actors and displayed the best.
frequent_combinations[frequent_combinations['length'] >= 2].sort_values("support", ascending=False).head()

,support,itemsets,length
689,0.006,"(daniel radcliffe, rupert grint)",2
687,0.005,"(daniel radcliffe, emma watson)",2
774,0.005,"(daniel radcliffe, rupert grint, emma watson)",3
701,0.005,"(rupert grint, emma watson)",2
723,0.004,"(robert de niro, joe pesci)",2


**Interpretation:** The maximum support is 0.006, corresponding to Daniel Radcliffe and Rupert Grind, it means that they appeared together in 6 films. It is present a length 3 with support 0.005 to the same previous actors and Emma Watson. This trio happened automatically because in Pass 2, the algorithm noticed that:


*   Daniel and Rupert are a frequent pair (6 movies).

*   Daniel and Emma are a frequent pair (5 movies).


*   Rupert and Emma are a frequent pair (5 movies).

Since all subsets were frequent, A-Priori generated the 3-actor candidate in its third pass and confirmed that they do, in fact, exceed the 0.002 threshold.

In [15]:
frequent_combinations[frequent_combinations['length'] >= 4].sort_values("support", ascending=False).head()

NameError: name 'frequent_combinations' is not defined

In [ ]:
frequent_combinations[frequent_combinations['length'] >= 5].shape

(0, 3)

As expected the maximum combinations are with length 4, there are no combinations with length equal o higher than 5. After identifying frequent itemsets of actors using Apriori, it is retrieved the corresponding films by filtering the original dataset and selecting only records containing the actors of the itemset.

In [ ]:
target_stars = ['viggo mortensen', 'elijah wood', 'ian mckellen', 'orlando bloom']
cols = ['Star1','Star2','Star3','Star4']
result = data[data[cols].apply(lambda r: set(target_stars).issubset(set(r)), axis=1)]
result[['Series_Title'] + cols]

,Series_Title,Star1,Star2,Star3,Star4
5,The Lord of the Rings: The Return of the King,elijah wood,viggo mortensen,ian mckellen,orlando bloom
13,The Lord of the Rings: The Two Towers,elijah wood,ian mckellen,viggo mortensen,orlando bloom


The itemset of the above mentioned actors are consireded in exactly 2 films of The Lord of the Rings.

# **Generating Association Rules**

Until now the central concept is about the support (s) that is applied over Itemsets (ej: {Robert De Niro, Joe Pesci}), and answers the question: How common is this group?
The next central concept is Confidence (c) that is applied over implications (ej: Robert De Niro -> Joe Pesci), and answers the quesion: How reliable is this relationship?

Considered these concepts associated with the project:
*   Confidence: The fraction of films containing actor A (I) that also contain actor B (j).
$$\text{confidence}(I \rightarrow j) = \frac{\text{support}(I \cup \{j\})}{\text{support}(I)}$$

*   Lift (Interest): How much more likely B is to appear given that A is present, compared to B's overall popularity.









In [ ]:
warnings.filterwarnings('ignore', category=DeprecationWarning)

from mlxtend.frequent_patterns import association_rules

# Generate association rules based on a minimum confidence threshold with 0.50 (50%) to ensure the rules have a fairly high confidence
rules = association_rules(frequent_combinations, metric="confidence", min_threshold=0.50)

# Sort the rules by 'lift' (interest) and then by 'confidence' to see the strongest relationships first
rules_sorted = rules.sort_values(by=['lift', 'confidence'], ascending=[False, False])

# Display the top 4 most interesting rules
rules_sorted.head(4)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4,(anatoliy solonitsyn),(nikolay grinko),0.002,0.002,0.002,1.0,500.0,1.0,0.001996,inf,1.0,1.0,1.0,1.0
5,(nikolay grinko),(anatoliy solonitsyn),0.002,0.002,0.002,1.0,500.0,1.0,0.001996,inf,1.0,1.0,1.0,1.0
6,(milhem cortaz),(andré ramiro),0.002,0.002,0.002,1.0,500.0,1.0,0.001996,inf,1.0,1.0,1.0,1.0
7,(andré ramiro),(milhem cortaz),0.002,0.002,0.002,1.0,500.0,1.0,0.001996,inf,1.0,1.0,1.0,1.0


In [ ]:
rules_sorted.shape

(337, 14)

*"The form of an association rule is I → j, where I is a set of items
and j is an item. The implication of this association rule is that if all of the
items in I appear in some basket, then j is “likely” to appear in that basket as
well."* (book 6.1.3)

**Interpretation:**
The first three rules show a Confidence of 1.0 and a very high Lift of 500. Anatoliy Solonitsyn -> Nikolay Grinko: This rule has 100% confidence. This means that in every single film in this dataset where Anatoliy Solonitsyn (I) appears, Nikolay Grinko (j) is also present. These rules have a Lift of 500. Significance: A high lift (interest) indicates that these actors are not just popular individually, but that their presence is heavily "caused" by or correlated with the other.

In [ ]:
rules_sorted.tail(4)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
119,(joe russo),(scarlett johansson),0.004,0.009,0.002,0.500000,55.555556,1.0,0.001964,1.982,0.985944,0.181818,0.495459,0.361111
95,(franka potente),(matt damon),0.004,0.011,0.002,0.500000,45.454545,1.0,0.001956,1.978,0.981928,0.153846,0.494439,0.340909
116,(joe pesci),(robert de niro),0.006,0.017,0.004,0.666667,39.215686,1.0,0.003898,2.949,0.980382,0.210526,0.660902,0.450980
2,(diane keaton),(al pacino),0.006,0.013,0.003,0.500000,38.461538,1.0,0.002922,1.974,0.979879,0.187500,0.493414,0.365385


**Interpretation:**
Joe Pesci -> Robert De Niro: This rule has a Confidence of 0.666667. This implies that if Joe Pesci is in the movie, there is a 66.66 chance that Robert De Niro is also in it. Chris Evans -> Scarlett Johansson: This rule has a Confidence of 0.50. This means that in half of the movies where Chris Evans appears, Scarlett Johansson is also a "Star" in the basket. Notice that the lift here is lower (around 39.2 to 55.5) compared to the 500.0 at the top list. This is because Robert De Niro and Scarlett Johansson are very frequent items on their own. Since they appear in many movies, the "surprise" of finding them with a specific partner is mathematically lower than with less frequent actors.

**Conclusion**:
The Apriori algorithm successfully extracted frequent actor combinations and meaningful association rules from the dataset. The experiment demonstrated both the practical applicability of frequent pattern mining and the importance of parameter tuning, particularly the support threshold, for obtaining informative results.

Although Apriori is effective for discovering frequent itemsets, it requires multiple passes over the dataset and can become computationally expensive as the number of items increases. This limitation motivates the use of more scalable algorithms such as SON or Toivonen's Algorithm, which will be explored in subsequent experiments.